In [41]:
import pandas as pd

import pandas as pd

ball = pd.read_csv(r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Data\ball_by_ball_data.csv")
matches = pd.read_csv(r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Data\ipl_matches_data.csv")
players = pd.read_csv(r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Data\players-data-updated.csv")
teams = pd.read_csv(r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Data\teams_data.csv")

ball["over_number"] = ball["over_number"] + 1

# Phase

ball["phase"] = ball["over_number"].apply(
    lambda x: "Powerplay"
    if x <= 6
    else (
        "Middle"
        if x <= 15
        else "Death"
    )
)
# Boundary Flag

ball["boundary_flag"] = (
    ball["batter_runs"].isin([4, 6])
).astype(int)

# Dot Ball Flag

ball["dot_ball_flag"] = (
    ball["total_runs"] == 0
).astype(int)

# Wicket Flag

ball["wicket_flag"] = (
    ball["is_wicket"] == 1
).astype(int)


ball[
    ["over_number","phase"]
].drop_duplicates().sort_values("over_number")

ball.to_csv(
    r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Data\clean_ball_data.csv",
    index=False
)


#Powerplay Summary
pp = ball[
    ball["phase"] == "Powerplay"
]

pp_summary = pp.groupby(
    ["match_id","innings"]
).agg(
    pp_runs=("total_runs","sum"),
    pp_wickets=("wicket_flag","sum")
).reset_index()

pp_summary.head()


# Middle Overs Summary

middle = ball[
    ball["phase"] == "Middle"
]

middle_summary = middle.groupby(
    ["match_id","innings"]
).agg(
    middle_runs=("total_runs","sum"),
    middle_wickets=("wicket_flag","sum")
).reset_index()

middle_summary.head()

#Death Overs Summary
death = ball[
    ball["phase"] == "Death"
]

death_summary = death.groupby(
    ["match_id","innings"]
).agg(
    death_runs=("total_runs","sum"),
    death_wickets=("wicket_flag","sum")
).reset_index()

death_summary.head()


#Boundary Summary
boundary_summary = ball.groupby(
    ["match_id","innings"]
).agg(
    boundaries=("boundary_flag","sum")
).reset_index()

boundary_summary.head()


#Dot Ball Summary
dotball_summary = ball.groupby(
    ["match_id","innings"]
).agg(
    dot_balls=("dot_ball_flag","sum")
).reset_index()

dotball_summary.head()

#Master Match Summary by merging

match_summary = pp_summary.merge(
    middle_summary,
    on=["match_id","innings"],
    how="left"
)

match_summary = match_summary.merge(
    death_summary,
    on=["match_id","innings"],
    how="left"
)

match_summary = match_summary.merge(
    boundary_summary,
    on=["match_id","innings"],
    how="left"
)

match_summary = match_summary.merge(
    dotball_summary,
    on=["match_id","innings"],
    how="left"
)

match_summary = match_summary.fillna(0)

match_summary.head()
 
#Add Match Information

match_info = matches[
    [
        "match_id",
        "season",
        "venue",
        "team1",
        "team2",
        "match_winner"
    ]
]

match_summary = match_summary.merge(
    match_info,
    on="match_id",
    how="left"
)

match_summary.head()

#Adding Batting Team

innings_team = ball.groupby(
    ["match_id","innings"]
)["team_batting"].first().reset_index()

match_summary = match_summary.merge(
    innings_team,
    on=["match_id","innings"],
    how="left"
)

match_summary.head()


#Creating Win Flag

match_summary["win_flag"] = (
    match_summary["team_batting"]
    ==
    match_summary["match_winner"]
).astype(int)


match_summary[
    [
        "match_id",
        "innings",
        "team_batting",
        "match_winner",
        "win_flag"
    ]
].head(10)

match_summary.shape

match_summary.to_csv(
    r"C:\Users\ckcha\OneDrive\Desktop\CSK_Dashboard\Clean_Data\match_summary.csv",
    index=False
)

,match_id,innings,pp_runs,pp_wickets
0,335982,1,61,1
1,335982,2,26,4
2,335983,1,53,1
3,335983,2,63,1
4,335984,1,40,2
